In [12]:
import pandas as pd
import datetime as dt
from datetime import timedelta
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [13]:
df = pd.read_csv('../Datasets/BOULDER_Electric_Vehicle_Charging_Station_Data.csv')
df.shape

(148136, 17)

In [14]:
print(df['Address'].nunique())
df['Address'].unique()

27


array(['2280 Junction Pl', '1275 Alpine Ave', '900 Baseline Rd',
       '1745 14th street', '1500 Pearl St', '1770 13th St',
       '1360 Gillaspie Dr', '1100 Spruce St', '900 Walnut St',
       '1400 Walnut St', '2052 Junction Pl', '1100 Walnut',
       '7315 Red Deer Dr', '3172 Broadway', '1739 Broadway',
       '2150 13th St', '5660 Sioux Dr', '5565 51st St', '1505 30th St',
       '3335 Airport Rd', '600 Baseline Rd', '5050 Pearl St',
       '2667 Broadway', '2240 Broadway', '5333 Valmont Rd',
       '2520 55th St', '5064 Pearl St'], dtype=object)

In [15]:
df['Address'].value_counts()

1505 30th St         20987
3172 Broadway        16278
1360 Gillaspie Dr    12349
900 Walnut St        11695
1100 Walnut           9471
1100 Spruce St        9194
1500 Pearl St         9060
5660 Sioux Dr         9056
600 Baseline Rd       8696
5333 Valmont Rd       8579
1770 13th St          6671
900 Baseline Rd       4762
2052 Junction Pl      3706
1400 Walnut St        3546
1745 14th street      2305
1739 Broadway         2165
5565 51st St          2052
3335 Airport Rd       2012
2667 Broadway         1667
2280 Junction Pl      1352
1275 Alpine Ave       1086
7315 Red Deer Dr       503
2520 55th St           394
2240 Broadway          276
5050 Pearl St           94
2150 13th St            90
5064 Pearl St           90
Name: Address, dtype: int64

In [16]:
df.columns

Index(['ObjectId2', 'Station_Name', 'Address', 'City', 'State_Province',
       'Zip_Postal_Code', 'Start_Date___Time', 'Start_Time_Zone',
       'End_Date___Time', 'End_Time_Zone', 'Total_Duration__hh_mm_ss_',
       'Charging_Time__hh_mm_ss_', 'Energy__kWh_', 'GHG_Savings__kg_',
       'Gasoline_Savings__gallons_', 'Port_Type', 'ObjectID'],
      dtype='object')

### **Remove Outliers Based on Charging Duration (paper method)**

Outliers are defined as charging durations that deviate more than three standard deviations from the median charging duration. This method helps in refining the dataset by excluding extreme values.


In [17]:
# Convert 'Charging_Time__hh_mm_ss_' to Timedelta
df['Charging_Time__hh_mm_ss_'] = pd.to_timedelta(df['Charging_Time__hh_mm_ss_'])

# Convert Timedelta to minutes
df['Total_Duration_minutes'] = df['Charging_Time__hh_mm_ss_'].dt.total_seconds() / 60

# Calculate the median and standard deviation of the duration in minutes
median_duration = df['Total_Duration_minutes'].median()
std_duration = df['Total_Duration_minutes'].std()

# Define the threshold for outliers
upper_threshold = median_duration + 3 * std_duration
lower_threshold = median_duration - 3 * std_duration

# Filter out the outliers
filtered_df = df[(df['Total_Duration_minutes'] >= lower_threshold) & (df['Total_Duration_minutes'] <= upper_threshold)]

# Check the percentage of rows removed
percent_removed = (1 - len(filtered_df) / len(df)) * 100
print(f'{percent_removed:.2f}% of outliers were removed.')

2.20% of outliers were removed.


In [18]:
filtered_df.head()

,ObjectId2,Station_Name,Address,City,State_Province,Zip_Postal_Code,Start_Date___Time,Start_Time_Zone,End_Date___Time,End_Time_Zone,Total_Duration__hh_mm_ss_,Charging_Time__hh_mm_ss_,Energy__kWh_,GHG_Savings__kg_,Gasoline_Savings__gallons_,Port_Type,ObjectID,Total_Duration_minutes
0,1,BOULDER / JUNCTION ST1,2280 Junction Pl,Boulder,Colorado,80301,1/1/2018 17:49,MDT,1/1/2018 19:52,MDT,2:03:02,0 days 02:02:44,6.504,2.732,0.816,Level 2,0,122.733333
1,2,BOULDER / JUNCTION ST1,2280 Junction Pl,Boulder,Colorado,80301,1/2/2018 8:52,MDT,1/2/2018 9:16,MDT,0:24:34,0 days 00:24:19,2.481,1.042,0.311,Level 2,1,24.316667
2,3,BOULDER / JUNCTION ST1,2280 Junction Pl,Boulder,Colorado,80301,1/2/2018 21:11,MDT,1/3/2018 6:23,MDT,9:12:21,0 days 03:40:52,15.046,6.319,1.888,Level 2,2,220.866667
3,4,BOULDER / ALPINE ST1,1275 Alpine Ave,Boulder,Colorado,80304,1/3/2018 9:19,MDT,1/3/2018 11:14,MDT,1:54:51,0 days 01:54:29,6.947,2.918,0.872,Level 2,3,114.483333
4,5,BOULDER / BASELINE ST1,900 Baseline Rd,Boulder,Colorado,80302,1/3/2018 14:13,MDT,1/3/2018 14:30,MDT,0:16:58,0 days 00:16:44,1.800,0.756,0.226,Level 2,4,16.733333


In [19]:
# Convert 'Start_Date___Time' and 'End_Date___Time' to datetime to get the day of the week
filtered_df['Start_Date___Time'] = pd.to_datetime(filtered_df['Start_Date___Time'])
filtered_df['End_Date___Time'] = pd.to_datetime(filtered_df['End_Date___Time'])

# # Extract the name of the day
# filtered_df['NameOfDay'] =  filtered_df['Start_Date___Time'].dt.day_name()

# # Adjust DayOfWeek to start with Sunday=0, Monday=1, ..., Saturday=6
# filtered_df['DayOfWeek'] = (filtered_df['Start_Date___Time'].dt.dayofweek + 1) % 7

# # Determine if it is a weekend
# filtered_df['IsWeekend'] = filtered_df['DayOfWeek'].apply(lambda x: 1 if x in [0, 6] else 0)

# filtered_df[['Address', 'Start_Date___Time', 'End_Date___Time', 'Energy__kWh_', 'NameOfDay', 'DayOfWeek', 'IsWeekend']].head()

### **Generate the Features in the Charging Data for Each Address**

1. **Time of Day (t)**:
   - **Description**: Represents the time index for each 10-minute interval in a 24-hour period.
   - **Values**: An integer ranging from 1 to 144, where each value corresponds to a specific 10-minute interval throughout the day. For example, 1 represents the first 10-minute interval of the day, and 144 represents the last.

2. **Day of the Week (d)**:
   - **Description**: Indicates the day of the week for the given interval.
   - **Values**: An integer where Sunday is 0, Monday is 1, ..., and Saturday is 6. This helps in understanding the temporal patterns related to specific days.

3. **Weekday/Weekend (w)**:
   - **Description**: Indicates whether the interval falls on a weekday or weekend.
   - **Values**: 1 if the interval is on a weekend (Saturday or Sunday), and 0 if it is on a weekday (Monday through Friday). This feature differentiates between the charging patterns observed during weekends versus weekdays.

4. **Average Charging Occupancy Rate Profile (p)**:
   - **Description**: Represents the average tendency of the charging occupancy rate for each 10-minute interval, specific to weekdays and weekends.
   - **Values**: Two constant vectors of 144 continuous variables each. One vector is for weekdays, and the other is for weekends. These vectors are calculated from the training dataset, with the average occupancy rate profile reflecting typical charging behaviors.

5. **Past Charging Occupancy States (y)**:
   - **Description**: Provides historical information on charging occupancy states for a given interval, showing past patterns.
   - **Values**: A sequence of binary values (0 or 1) representing whether charging was occurring during the previous k steps. For example, `y_t-1` indicates the occupancy state one interval before `t`, `y_t-2` two intervals before, and so on. This sequence helps in understanding historical trends and predicting future states based on past data.

These features collectively enable a comprehensive analysis of charging behavior, incorporating time-based, day-based, and historical occupancy data for more accurate predictions and insights.


    Create DataFrames for Each Address

In [20]:
# Group by Address
grouped = filtered_df.groupby('Address')
columns = ['Start_Date___Time', 'End_Date___Time']

# Create a dictionary where keys are Address and values are DataFrames of each Address
Address_dict = {Address: group.drop(columns='Address') for Address, group in grouped}

# Assuming Address_dict is already defined
for idx, (Address, df) in enumerate(Address_dict.items(), start=1):
    globals()[f'df_{idx}'] = df[columns]
    #globals()[f'df_{idx}'] = df
    print(len(globals()[f'df_{idx}']))

9171
9412
1006
11945
3295
8783
20617
1959
2195
6583
3604
90
276
1332
349
1562
15881
1918
92
55
8459
2044
8968
8550
441
4758
11537


##### **Process Charging Data Function**

The `process_charging_data` function processes charging transaction data by generating time intervals and adding relevant features. Here’s a breakdown of its functionality:

1. **Extract Unique Dates**:
   - Identify all unique dates from the 'Start_Date___Time' column of the input DataFrame `df_transactions`.

2. **Generate 10-Minute Intervals**:
   - Define a helper function `generate_intervals` to create 10-minute intervals for each date. This function returns the start and end times of these intervals.

3. **Create Interval DataFrames**:
   - Iterate over each unique date to generate intervals and create a DataFrame for each date. This DataFrame includes the interval start and end times and a sequential time index (`t`).

4. **Add Transaction Indicator**:
   - Define the `add_transaction_y_indicator` function to check if any transactions overlap with each interval. Add a binary column 'y' indicating the presence (1) or absence (0) of transactions.

5. **Add Previous State Indicator**:
   - Use the `past_chg_occ_state` function to add a column 'y_t_1' representing the previous state of the transaction indicator ('y').

6. **Add Weekdays and Weekends Information**:
   - In the `add_weekdays_and_weekends` function:
     - Convert 'Start_Date___Time' to datetime and extract the name of the day and the adjusted day of the week.
     - Determine if the date falls on a weekend and add a 'weekend' column.

In [21]:
def process_charging_data(df_transactions):
    # Extract unique dates by removing the time component
    df_transactions['Date'] = pd.to_datetime(df_transactions['Start_Date___Time']).dt.date
    unique_dates = df_transactions[['Date']].drop_duplicates()

    # Function to generate 10-minute intervals for a given date
    def generate_intervals(date):
        start_time = pd.Timestamp(date)
        intervals_start = pd.date_range(start=start_time, periods=144, freq='10T')
        intervals_end = intervals_start + pd.Timedelta(minutes=10)
        return intervals_start, intervals_end

    # Generate the intervals for all unique dates and create the DataFrame
    intervals_list = []

    for idx, row in unique_dates.iterrows():
        date = row['Date']
        intervals_start, intervals_end = generate_intervals(date)
        data_chg = pd.DataFrame({
            'Date': date,
            'IntervalStart': intervals_start,
            'IntervalEnd': intervals_end,
            't': np.arange(1, 145)
        })
        intervals_list.append(data_chg)

    # Concatenate all the individual DataFrames into one
    data_chg = pd.concat(intervals_list).reset_index(drop=True)

    # Add transaction indicator
    def add_transaction_y_indicator(intervals_df, transactions_df):
        y_values = []

        for idx, row in intervals_df.iterrows():
            interval_start = row['IntervalStart']
            interval_end = row['IntervalEnd']
            
            # Check if any transaction falls within the interval
            transaction_exists = transactions_df[
                (transactions_df['Start_Date___Time'] < interval_end) & 
                (transactions_df['End_Date___Time'] > interval_start)
            ].shape[0] > 0
            
            y_values.append(1 if transaction_exists else 0)
        
        intervals_df['y'] = y_values
        return intervals_df
    
    def past_chg_occ_state(data_chg, df_transactions):
        data_chg = add_transaction_y_indicator(data_chg, df_transactions)
    
        # Add the y_t_1 column
        data_chg['y_t_1'] = data_chg['y'].shift(1).astype('Int64')

        return data_chg

    def add_weekdays_and_weekends(data_chg, df_transactions):
        data_chg = past_chg_occ_state(data_chg, df_transactions)

        # Convert 'Start_Date___Time' and 'EndDate' to datetime to get the day of the week
        data_chg['Start_Date___Time'] = pd.to_datetime(data_chg['IntervalStart'])  # Update to use IntervalStart
        # Extract the name of the day
        data_chg['NameOfDay'] =  data_chg['Start_Date___Time'].dt.day_name()
        # Adjust DayOfWeek to start with Sunday=0, Monday=1, ..., Saturday=6
        data_chg['dayofweek'] = (data_chg['Start_Date___Time'].dt.dayofweek + 1) % 7
        # Determine if it is a weekend
        data_chg['weekend'] = data_chg['dayofweek'].apply(lambda x: 1 if x in [0, 6] else 0)

        return data_chg

    data_chg = add_weekdays_and_weekends(data_chg, df_transactions).dropna()[:-144]

    print(len(data_chg) + 1)
    return data_chg

    Create and store the Dataframes is csv files

In [22]:
# Loop through 27 DataFrames and save each to a CSV file
for i in range(1, len(Address_dict)+1):  # 1 to 27 inclusive
    # Construct the DataFrame variable name dynamically
    df_variable = globals()[f'df_{i}']
    
    # Process the DataFrame using your function
    data_chg = process_charging_data(df_variable)
    
    # Save the processed DataFrame to a CSV file
    data_chg[['t', 'dayofweek', 'weekend', 'y_t_1', 'y']].to_csv(f'../Datasets/occupancy_data/ChargingData/data_chg_{i}.csv', index=False)

224352
231840
46800
243648
149328
226080
171792
95040
94032
199584
130464
4608
16272
56592
18000
65664
262800
88272
5472
4176
101232
72000
191664
127296
25056
148896
237024


In [47]:
data_chg

,Date,IntervalStart,IntervalEnd,t,y,y_t_1,Start_Date___Time,NameOfDay,dayofweek,weekend
1,2018-01-25,2018-01-25 00:10:00,2018-01-25 00:20:00,2,0,0,2018-01-25 00:10:00,Thursday,4,0
2,2018-01-25,2018-01-25 00:20:00,2018-01-25 00:30:00,3,0,0,2018-01-25 00:20:00,Thursday,4,0
3,2018-01-25,2018-01-25 00:30:00,2018-01-25 00:40:00,4,0,0,2018-01-25 00:30:00,Thursday,4,0
4,2018-01-25,2018-01-25 00:40:00,2018-01-25 00:50:00,5,0,0,2018-01-25 00:40:00,Thursday,4,0
5,2018-01-25,2018-01-25 00:50:00,2018-01-25 01:00:00,6,0,0,2018-01-25 00:50:00,Thursday,4,0
...,...,...,...,...,...,...,...,...,...,...
237019,2023-09-14,2023-09-14 23:10:00,2023-09-14 23:20:00,140,0,0,2023-09-14 23:10:00,Thursday,4,0
237020,2023-09-14,2023-09-14 23:20:00,2023-09-14 23:30:00,141,0,0,2023-09-14 23:20:00,Thursday,4,0
237021,2023-09-14,2023-09-14 23:30:00,2023-09-14 23:40:00,142,0,0,2023-09-14 23:30:00,Thursday,4,0
237022,2023-09-14,2023-09-14 23:40:00,2023-09-14 23:50:00,143,0,0,2023-09-14 23:40:00,Thursday,4,0


In [40]:
data_chg_1 = process_charging_data(df_1)
data_chg_2 = process_charging_data(df_2)
data_chg_3 = process_charging_data(df_3)
data_chg_4 = process_charging_data(df_4)
data_chg_5 = process_charging_data(df_5)
data_chg_6 = process_charging_data(df_6)
data_chg_7 = process_charging_data(df_7)
data_chg_8 = process_charging_data(df_8)
data_chg_9 = process_charging_data(df_9)
data_chg_10 = process_charging_data(df_10)
data_chg_11 = process_charging_data(df_11)
data_chg_12 = process_charging_data(df_12)
data_chg_13 = process_charging_data(df_13)
data_chg_14 = process_charging_data(df_14)
data_chg_15 = process_charging_data(df_15)
data_chg_16 = process_charging_data(df_16)
data_chg_17 = process_charging_data(df_17)
data_chg_18 = process_charging_data(df_18)
data_chg_19 = process_charging_data(df_19)
data_chg_20 = process_charging_data(df_20)
data_chg_21 = process_charging_data(df_21)
data_chg_22 = process_charging_data(df_22)
data_chg_23 = process_charging_data(df_23)
data_chg_24 = process_charging_data(df_24)
data_chg_25 = process_charging_data(df_25)
data_chg_26 = process_charging_data(df_26)
data_chg_27 = process_charging_data(df_27)

224352
231840
46800
243648
149328
226080
171792
95040
94032
199584
130464
4608
16272
56592
18000
65664
262800
88272
5472
4176
101232
72000
191664
127296
25056
148896
237024


    Ensure that the process_charging_data function is working correctly and does not create any duplicate entries.

In [43]:
# Check for duplicates based on the IntervalStart and IntervalEnd columns
def check_duplicates(df, data_chg_name):
    duplicates = df[df.duplicated(subset=['IntervalStart', 'IntervalEnd'])]

    if not duplicates.empty:
        print(data_chg_name)
        print("Duplicate rows found based on 'IntervalStart' and 'IntervalEnd':",len(duplicates))
        print(duplicates)
        
    else:
        print("No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.")


# Loop through the range of 1 to 27 to dynamically check duplicates for each DataFrame
for i in range(1, len(Address_dict)+1):
    # Construct the DataFrame variable name dynamically
    data_chg_variable = globals()[f'data_chg_{i}']
    
    # Check for duplicates
    check_duplicates(data_chg_variable, f'data_chg_{i}')

No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart' and 'IntervalEnd'.
No duplicate rows found based on 'IntervalStart'

##### **Calculate Occupancy Rate Function**

The `calculate_occupancy_rate` function computes the average charging occupancy rate profiles for weekdays and weekends based on the given DataFrame. It produces two key features: the average occupancy rate profile for weekdays and the average occupancy rate profile for weekends.

1. **Filter DataFrame**:
   - **Weekends**: Select rows where 'NameOfDay' is either 'Saturday' or 'Sunday'.
   - **Weekdays**: Select rows where 'NameOfDay' is not 'Saturday' or 'Sunday'.

2. **Calculate Mean for Weekdays**:
   - Pivot the `weekdays` DataFrame with 'Start_Date___Time' as the index, 't' (time index) as columns, and 'y' (charging occupancy state) as values.
   - Fill missing values with 0.
   - Calculate the mean of the occupancy values for each time interval across all weekdays.

3. **Calculate Mean for Weekends**:
   - Similarly, pivot the `weekends` DataFrame and compute the mean of the occupancy values for each time interval across all weekends.

4. **Combine Mean Values**:
   - Create a DataFrame `data_chg_pred_occ_t` with two columns: 'weekday' and 'weekend', containing the mean values for each time interval.
   - Reset the index to convert the DataFrame to a more standard format.

5. **Return the Result**:
   - Return the resulting DataFrame with columns 'weekday' and 'weekend', excluding the time index column 't'.

In [44]:
def calculate_occupancy_rate(df):
    # Filter DataFrame
    weekends = df[df['NameOfDay'].isin(['Saturday', 'Sunday'])]
    weekdays = df[~df['NameOfDay'].isin(['Saturday', 'Sunday'])]

    # Pivot and calculate mean for weekdays
    mean_y_weekday = weekdays.pivot(index='Start_Date___Time', columns='t', values='y').fillna(0).mean()

    # Pivot and calculate mean for weekends
    mean_y_weekend = weekends.pivot(index='Start_Date___Time', columns='t', values='y').fillna(0).mean()

    # Combine the mean values into a DataFrame
    data_chg_pred_occ_t = pd.DataFrame({
        'weekday': mean_y_weekday,
        'weekend': mean_y_weekend
    }).reset_index()

    return data_chg_pred_occ_t.drop(columns='t')

In [45]:
# Loop through 27 DataFrames and save each to a CSV file
for i in range(1, len(Address_dict)+1):  # 1 to 27 inclusive
    # Construct the DataFrame variable name dynamically
    df_variable = globals()[f'data_chg_{i}']
    
    # Process the DataFrame using your function
    data_chg_pred_occ_t = calculate_occupancy_rate(df_variable)
    
    # Save the processed DataFrame to a CSV file
    data_chg_pred_occ_t.to_csv(f'../Datasets/occupancy_data/RateOfChargingData/data_chg_pred_occ_t_{i}.csv', index=False)

In [46]:
data_chg_pred_occ_t

,weekday,weekend
0,0.000186,0.000608
1,0.000168,0.000552
2,0.000156,0.000537
3,0.000144,0.000495
4,0.000132,0.000495
...,...,...
139,0.000409,0.000537
140,0.000373,0.000481
141,0.000361,0.000467
142,0.000337,0.000438
